In [ ]:
import sys
import os
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive', force_remount=True)

!git clone https://github.com/mahithapen/timeseries-transformer.git

%cd timeseries-transformer

REPO_ROOT = Path('/content/timeseries-transformer')
CODE_ROOT = REPO_ROOT / 'code'

DATA_DIR = Path('/content/drive/MyDrive/DLFinalProject/data')

DRIVE_SAVE_DIR = Path('/content/drive/MyDrive/DLFinalProject/runs')
CHECKPOINT_DIR = DRIVE_SAVE_DIR / 'checkpoints'
RESULTS_DIR = DRIVE_SAVE_DIR / 'results'
SUMMARY_PATH = RESULTS_DIR / 'summary.csv'

CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

if str(CODE_ROOT) not in sys.path:
    sys.path.insert(0, str(CODE_ROOT))

!pip install -r code/requirements.txt

In [ ]:
import torch

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

DATASETS = {
    'weather': DATA_DIR / 'weather.csv',
    'electricity': DATA_DIR / 'electricity.csv',
    'traffic': DATA_DIR / 'traffic.csv',
    'ithaca_weather': DATA_DIR / 'ithaca_weather.csv', # Added for the new dataset
}

PRED_LENS = [96]

MODEL_VARIANTS = {
    'dlinear': {
        'model_type': 'dlinear',
        'seq_len': 336,
        'patch_len': 16,   # Ignored
        'stride': 8,       # Ignored
        'padding_patch': 'end', # Ignored
    },
    'patchtst42': {
        'model_type': 'patchtst',
        'seq_len': 336,
        'patch_len': 16,
        'stride': 8,
        'padding_patch': 'end',
    },
    'patchtst64': {
        'model_type': 'patchtst',
        'seq_len': 512,
        'patch_len': 16,
        'stride': 8,
        'padding_patch': 'end',
    },
}

COMMON_MODEL_CONFIG = {
    'd_model': 128, 'n_heads': 16, 'n_layers': 3, 'd_ff': 256,
    'dropout': 0.2, 'attn_dropout': 0.0, 'fc_dropout': 0.2, 'head_dropout': 0.0,
}

TRAIN_CONFIG = {
    'epochs': 100, 'batch_size': 128, 'lr': 1e-4,
    'scheduler': 'type3', 'patience': 20, 'seed': 42,
}

DATASET_BATCH_SIZES = {'weather': 128, 'electricity': 32, 'traffic': 24, 'ithaca_weather': 128}
CPU_DATASET_BATCH_SIZES = {'weather': 64, 'electricity': 8, 'traffic': 4, 'ithaca_weather': 64}

In [ ]:
import csv
import subprocess

from dataclasses import fields
import pandas as pd
from torch.utils.data import DataLoader

from data.window_dataset import build_datasets
from models.patchtst import PatchTST, PatchTSTConfig
from models.dlinear import DLinear

SUMMARY_COLUMNS = ['dataset', 'task', 'variant', 'seq_len', 'pred_len', 'test_mae', 'test_mse', 'checkpoint']

def normalize_summary_frame(summary):
    for column in SUMMARY_COLUMNS:
        if column not in summary.columns:
            summary[column] = ''
    return summary[SUMMARY_COLUMNS]

def read_summary():
    if SUMMARY_PATH.exists():
        return normalize_summary_frame(pd.read_csv(SUMMARY_PATH))
    return pd.DataFrame(columns=SUMMARY_COLUMNS)

def already_done(dataset_name, task_name, variant_name, pred_len, checkpoint_path):
    if not checkpoint_path.exists() or not SUMMARY_PATH.exists():
        return False
    summary = read_summary()
    matches = summary[
        (summary['dataset'] == dataset_name) &
        (summary['task'] == task_name) &
        (summary['variant'] == variant_name) &
        (summary['pred_len'] == pred_len)
    ]
    return not matches.empty

def append_summary_row(row):
    file_exists = SUMMARY_PATH.exists()
    if file_exists:
        current_summary = read_summary()
        if list(pd.read_csv(SUMMARY_PATH, nrows=0).columns) != SUMMARY_COLUMNS:
            current_summary.to_csv(SUMMARY_PATH, index=False)
    with SUMMARY_PATH.open('a', newline='') as handle:
        writer = csv.DictWriter(handle, fieldnames=SUMMARY_COLUMNS)
        if not file_exists:
            writer.writeheader()
        writer.writerow(row)

def batch_size_for(dataset_name):
    if device == 'cpu':
        return CPU_DATASET_BATCH_SIZES.get(dataset_name, TRAIN_CONFIG['batch_size'])
    return DATASET_BATCH_SIZES.get(dataset_name, TRAIN_CONFIG['batch_size'])

def evaluate_checkpoint(checkpoint_path, data_path, dataset_name):
    checkpoint = torch.load(checkpoint_path, map_location=device)
    saved_cfg = checkpoint['config']
    is_dlinear = saved_cfg.get('model_type') == 'dlinear'

    seq_len = saved_cfg['seq_len']
    pred_len = saved_cfg['pred_len']

    bundle = build_datasets(
        data_path=data_path, seq_len=seq_len, pred_len=pred_len,
        val_ratio=checkpoint['val_ratio'], test_ratio=checkpoint['test_ratio'], scale=checkpoint['scale'],
    )

    loader = DataLoader(bundle.test, batch_size=batch_size_for(dataset_name), shuffle=False)

    if is_dlinear:
        model = DLinear(seq_len=seq_len, pred_len=pred_len, channels=checkpoint['in_channels']).to(device)
    else:
        config_fields = {field.name for field in fields(PatchTSTConfig)}

        patch_kwargs = {k: v for k, v in saved_cfg.items() if k in config_fields}

        config = PatchTSTConfig(**patch_kwargs)
        model = PatchTST(config, in_channels=checkpoint['in_channels']).to(device)

    state_dict = checkpoint['model_state_dict']
    clean_state_dict = {k.replace('_orig_mod.', ''): v for k, v in state_dict.items()}

    model.load_state_dict(clean_state_dict)
    model.eval()

    mae, mse = torch.nn.L1Loss(), torch.nn.MSELoss()
    total_mae, total_mse, count = 0.0, 0.0, 0

    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            pred = model(x)
            total_mae += mae(pred, y).item()
            total_mse += mse(pred, y).item()
            count += 1

    return {'test_mae': total_mae / max(1, count), 'test_mse': total_mse / max(1, count)}

def run_experiment(dataset_name, hierarchical=False):
    data_path = DATASETS[dataset_name]
    batch_size = batch_size_for(dataset_name)
    task_name = 'hierarchical' if hierarchical else 'supervised'
    rows = []

    if not data_path.exists():
        raise FileNotFoundError(f'Missing dataset file: {data_path}')

    for variant_name, variant_cfg in MODEL_VARIANTS.items():

        if hierarchical and variant_cfg['model_type'] == 'dlinear':

            print(f"Skipping {dataset_name} hierarchical run for DLinear (not supported).")

            continue


        for pred_len in PRED_LENS:
            checkpoint_path = CHECKPOINT_DIR / f'{dataset_name}_{task_name}_{variant_name}_seq{variant_cfg["seq_len"]}_pred{pred_len}.pt'

            if already_done(dataset_name, task_name, variant_name, pred_len, checkpoint_path):
                print(f'Skipping completed run: {dataset_name} {task_name} {variant_name} pred_len={pred_len}')
                continue

            cmd = [
                'python', '-u', 'code/train.py',
                '--model-type', variant_cfg['model_type'],
                '--data', str(data_path),
                '--seq-len', str(variant_cfg['seq_len']),
                '--pred-len', str(pred_len),
                '--patch-len', str(variant_cfg['patch_len']),
                '--stride', str(variant_cfg['stride']),
                '--padding-patch', variant_cfg['padding_patch'],
                '--d-model', str(COMMON_MODEL_CONFIG['d_model']),
                '--n-heads', str(COMMON_MODEL_CONFIG['n_heads']),
                '--n-layers', str(COMMON_MODEL_CONFIG['n_layers']),
                '--d-ff', str(COMMON_MODEL_CONFIG['d_ff']),
                '--dropout', str(COMMON_MODEL_CONFIG['dropout']),
                '--epochs', str(TRAIN_CONFIG['epochs']),
                '--batch-size', str(batch_size),
                '--lr', str(TRAIN_CONFIG['lr']),
                '--patience', str(TRAIN_CONFIG['patience']),
                '--seed', str(TRAIN_CONFIG['seed']),
                '--device', device,
                '--checkpoint', str(checkpoint_path),
                '--resume',
            ]
            if hierarchical:

                cmd += [

                    '--hierarchical-patching',

                    '--hierarchical-levels', '2',

                    '--hierarchical-merge-factor', '2',

                ]

            print(f'Running {dataset_name} {task_name} {variant_name} pred_len={pred_len} on {device}')
            process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

            for line in process.stdout:
                sys.stdout.write(line)
                sys.stdout.flush()

            process.wait()

            if process.returncode != 0:
                raise subprocess.CalledProcessError(process.returncode, cmd)

            metrics = evaluate_checkpoint(checkpoint_path, data_path, dataset_name)
            row = {
                'dataset': dataset_name,
                'task': task_name,
                'variant': variant_name,
                'seq_len': variant_cfg['seq_len'],
                'pred_len': pred_len,
                'test_mae': metrics['test_mae'],
                'test_mse': metrics['test_mse'],
                'checkpoint': str(checkpoint_path),
            }
            append_summary_row(row)
            rows.append(row)
            print(f'Saved metrics to summary in Drive!')

    return read_summary() if not rows else pd.DataFrame(rows)

In [ ]:
weather_supervised_df = run_experiment('weather', hierarchical=False)

weather_hierarchical_df = run_experiment('weather', hierarchical=True)

display(weather_hierarchical_df)

In [ ]:
electricity_supervised_df = run_experiment('electricity', hierarchical=False)

electricity_hierarchical_df = run_experiment('electricity', hierarchical=True)

display(electricity_hierarchical_df)

In [ ]:
traffic_supervised_df = run_experiment('traffic', hierarchical=False)

traffic_hierarchical_df = run_experiment('traffic', hierarchical=True)

display(traffic_hierarchical_df)

In [ ]:
checkpoint_path = CHECKPOINT_DIR / 'traffic_hierarchical_patchtst64_seq512_pred96.pt'
data_path = DATASETS['weather']

evaluate_checkpoint(checkpoint_path, data_path,'weather')

In [ ]:
checkpoint_path = CHECKPOINT_DIR / 'traffic_hierarchical_patchtst64_seq512_pred96.pt'
data_path = DATASETS['weather']

evaluate_checkpoint(checkpoint_path, data_path,'weather')

In [ ]:
all_results_df = read_summary()
all_results_df

In [ ]:
# UNFINISHED - ITHACA WEATHER DATASET PROCESSING
# Add 'ithaca_weather': DATA_DIR / 'ithaca_weather.csv' to datasets
import pandas as pd

df = pd.read_csv('raw_ithaca_weather.csv')

# 2. Pivot the table from Long to Wide format
# We use 'Date' as the index, 'feature' for the new column names,
# and 'prediction' for the actual values filling those columns.
wide_df = df.pivot(index='Date', columns='feature', values='prediction').reset_index()

# 3. Rename 'Date' to lowercase 'date' (standard convention for these models)
wide_df.rename(columns={'Date': 'date'}, inplace=True)

# 4. Optional: Sort by date to ensure strictly chronological order
wide_df['date'] = pd.to_datetime(wide_df['date'])
wide_df = wide_df.sort_values('date')

# 5. Save the cleaned, model-ready dataset
wide_df.to_csv('ithaca_weather.csv', index=False)

print(wide_df.head())

In [ ]:
ithaca_supervised_df = run_experiment('ithaca_weather', hierarchical=False)

ithaca_hierarchical_df = run_experiment('ithaca_weather', hierarchical=True)

display(ithaca_hierarchical_df)

### Calculate results from checkpoints

In [ ]:
import re

def rebuild_summary():
    if SUMMARY_PATH.exists():
        SUMMARY_PATH.unlink() # Delete existing summary file

    all_rows = []
    for checkpoint_file in CHECKPOINT_DIR.glob('*.pt'):
        file_name = checkpoint_file.name

        # Regex to parse the checkpoint filename
        match = re.match(r'([a-z_]+)_([a-z]+)_([a-z0-9]+)_seq(\d+)_pred(\d+).pt', file_name)
        if not match:
            print(f'Skipping malformed checkpoint filename: {file_name}')
            continue

        dataset_name, task_name, variant_name, seq_len, pred_len = match.groups()

        # Ensure correct type for seq_len and pred_len
        seq_len = int(seq_len)
        pred_len = int(pred_len)

        # Removed the skipping condition for patchtst42

        # Check if the dataset name is valid based on DATASETS keys
        if dataset_name not in DATASETS:
            print(f'Skipping checkpoint {file_name} as dataset \'{dataset_name}\' is not defined in DATASETS.')
            continue

        data_path = DATASETS[dataset_name]

        try:
            print(f'Evaluating checkpoint: {file_name}')
            metrics = evaluate_checkpoint(checkpoint_file, data_path, dataset_name)
            row = {
                'dataset': dataset_name,
                'task': task_name,
                'variant': variant_name,
                'seq_len': seq_len,
                'pred_len': pred_len,
                'test_mae': metrics['test_mae'],
                'test_mse': metrics['test_mse'],
                'checkpoint': str(checkpoint_file),
            }
            append_summary_row(row)
            all_rows.append(row)
        except Exception as e:
            print(f'Error evaluating checkpoint {file_name}: {e}')

    return normalize_summary_frame(pd.DataFrame(all_rows))

print('Rebuilding summary from all checkpoints...')
rebuilt_df = rebuild_summary()
display(rebuilt_df)

In [ ]:
sorted_df = rebuilt_df.sort_values(by=['dataset', 'task', 'variant'], ascending=[False, False, True])
display(sorted_df)

In [ ]:
import pandas as pd
import numpy as np

# Helper to map task and variant to the display name
def map_model_name(task, variant):
    if task == 'supervised' and variant == 'dlinear': return 'dLinear'
    if task == 'supervised' and variant == 'patchtst42': return 'PatchTST/42 (L=336)'
    if task == 'supervised' and variant == 'patchtst64': return 'PatchTST/64 (L=512)'
    return None

df_clean = rebuilt_df.copy()
df_clean['Model_Name'] = df_clean.apply(lambda r: map_model_name(r['task'], r['variant']), axis=1)
df_clean = df_clean.dropna(subset=['Model_Name'])

# Pivot the table
pivot = df_clean.pivot_table(index='Model_Name', columns='dataset', values=['test_mse', 'test_mae'])

# Order columns specifically: Weather, Electricity, Traffic with MSE then MAE
datasets = ['weather', 'electricity', 'traffic']
new_cols = []
for d in datasets:
    new_cols.extend([('test_mse', d), ('test_mae', d)])

# Reindex to get the correct column order, ignore missing columns gracefully
existing_cols = [c for c in new_cols if c in pivot.columns]
pivot = pivot[existing_cols]

# Rename column levels nicely
pivot.columns = pd.MultiIndex.from_tuples([(d.title(), 'MSE') if m == 'test_mse' else (d.title(), 'MAE') for m, d in pivot.columns], names=['Dataset', 'Metric'])

# Define row order for 'Ours'
row_order = ['dLinear', 'PatchTST/42 (L=336)', 'PatchTST/64 (L=512)', 'HPatch/42 (L=336)', 'HPatch/64 (L=512)']
existing_rows = [r for r in row_order if r in pivot.index]
pivot = pivot.reindex(existing_rows)

# Add 'Ours' level to index
pivot.index = pd.MultiIndex.from_product([['Ours'], pivot.index], names=['Model', 'Variant'])

# Create the 'Paper' baseline dataframe
paper_data = {
    ('Weather', 'MSE'): [0.152, 0.149],
    ('Weather', 'MAE'): [0.199, 0.198],
    ('Electricity', 'MSE'): [0.130, 0.129],
    ('Electricity', 'MAE'): [0.222, 0.222],
    ('Traffic', 'MSE'): [0.367, 0.367],
    ('Traffic', 'MAE'): [0.251, 0.249]
}
paper_idx = pd.MultiIndex.from_product([['Paper'], ['PatchTST/42 (L=336)', 'PatchTST/64 (L=512)']], names=['Model', 'Variant'])
paper_df = pd.DataFrame(paper_data, index=paper_idx)

# Ensure column matching before concatenation
paper_df = paper_df[pivot.columns]

# Combine and format to 4 decimal places
final_table = pd.concat([pivot, paper_df]).round(4)
display(final_table)